# Model Evaluation: Performance Assessment

Comprehensive evaluation including:
- Classification metrics (Accuracy, Precision, Recall, F1, AUC-ROC)
- Confusion matrices
- ROC & Precision-Recall curves
- Cross-validation stability
- Statistical significance testing

## Section 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import re, warnings, time, os, json
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, cross_val_score)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from scipy.sparse import hstack, csr_matrix
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('Libraries loaded.')

In [ ]:
df = pd.read_csv('../data/cumulative_ai_customer_communication_dataset.csv', low_memory=False)
df['issue_reported_at'] = pd.to_datetime(df['issue_reported_at'], errors='coerce', dayfirst=True)
df['issue_responded'] = pd.to_datetime(df['issue_responded'], errors='coerce', dayfirst=True)
df['target'] = (df['csat_score'] >= 4).astype(int)
print(f'Dataset: {df.shape[0]} rows, Target positive rate: {df["target"].mean()*100:.1f}%')

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def preprocess_text(text):
    if pd.isna(text) or not isinstance(text, str): return ''
    text = text.lower()
    text = text.encode('ascii','ignore').decode('ascii')
    text = re.sub(r'[^a-z\s]','',text)
    text = re.sub(r'\s+',' ',text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t)>1]
    return ' '.join(tokens)
print('Preprocessing text...')
df['cleaned_message'] = df['customer_message'].apply(preprocess_text)
print(f'Done. Non-empty: {(df["cleaned_message"]!="").sum()}')

In [ ]:
df['response_time_minutes'] = ((df['issue_responded']-df['issue_reported_at']).dt.total_seconds()/60).clip(lower=0).fillna(0)
df['issue_hour'] = df['issue_reported_at'].dt.hour.fillna(0).astype(int)
df['issue_day_of_week'] = df['issue_reported_at'].dt.dayofweek.fillna(0).astype(int)
for col,src in [('channel_encoded','channel_name'),('category_encoded','category'),
                ('subcategory_encoded','sub-category'),('shift_encoded','agent_shift')]:
    le=LabelEncoder(); df[col]=le.fit_transform(df[src].fillna('Unknown'))
tenure_map={'On Job Training':0,'0-30':1,'31-60':2,'61-90':3,'>90':4}
df['tenure_encoded']=df['tenure_bucket'].map(tenure_map).fillna(0).astype(int)
df['has_message']=(df['cleaned_message']!='').astype(int)
df['cleaned_word_count']=df['cleaned_message'].apply(lambda x:len(x.split()) if x else 0)
structured_features=['response_time_minutes','issue_hour','issue_day_of_week','channel_encoded',
    'category_encoded','subcategory_encoded','shift_encoded','tenure_encoded',
    'message_length','word_count','has_message','cleaned_word_count']
print(f'Structured features: {len(structured_features)}')

In [ ]:
X_structured = df[structured_features].fillna(0)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X_structured, y, test_size=0.2, random_state=42, stratify=y)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=5)
X_text_train = tfidf.fit_transform(df.loc[X_train.index,'cleaned_message'])
X_text_test = tfidf.transform(df.loc[X_test.index,'cleaned_message'])
X_combined_train = hstack([X_text_train, csr_matrix(X_train.values)])
X_combined_test = hstack([X_text_test, csr_matrix(X_test.values)])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg_count=(y_train==0).sum(); pos_count=(y_train==1).sum(); scale_weight=neg_count/pos_count
print(f'Train:{X_train.shape[0]}, Test:{X_test.shape[0]}, TF-IDF:{X_text_train.shape[1]}, Combined:{X_combined_train.shape[1]}')

## Section 2: Train All Models

In [ ]:
# Train models
print('Training models...')
lr_model=LogisticRegression(max_iter=1000,random_state=42,class_weight='balanced')
lr_model.fit(X_combined_train,y_train)
y_pred_lr=lr_model.predict(X_combined_test);y_prob_lr=lr_model.predict_proba(X_combined_test)[:,1]

rf_model=RandomForestClassifier(n_estimators=200,max_depth=20,random_state=42,class_weight='balanced',n_jobs=-1)
rf_model.fit(X_text_train,y_train)
y_pred_rf=rf_model.predict(X_text_test);y_prob_rf=rf_model.predict_proba(X_text_test)[:,1]

xgb_model=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,scale_pos_weight=scale_weight,random_state=42,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(X_train,y_train)
y_pred_xgb=xgb_model.predict(X_test);y_prob_xgb=xgb_model.predict_proba(X_test)[:,1]

svm_model=CalibratedClassifierCV(LinearSVC(max_iter=2000,random_state=42,class_weight='balanced'),cv=3)
svm_model.fit(X_text_train,y_train)
y_pred_svm=svm_model.predict(X_text_test);y_prob_svm=svm_model.predict_proba(X_text_test)[:,1]

gb_model=GradientBoostingClassifier(n_estimators=200,max_depth=5,learning_rate=0.1,random_state=42)
gb_model.fit(X_train,y_train)
y_pred_gb=gb_model.predict(X_test);y_prob_gb=gb_model.predict_proba(X_test)[:,1]

models_dict={'Logistic Regression':(y_pred_lr,y_prob_lr),'Random Forest':(y_pred_rf,y_prob_rf),
    'XGBoost':(y_pred_xgb,y_prob_xgb),'SVM':(y_pred_svm,y_prob_svm),'Gradient Boosting':(y_pred_gb,y_prob_gb)}
print(f'All {len(models_dict)} models trained.')

## Section 3: Metrics Table

In [ ]:
# Comprehensive metrics
print('=== MODEL EVALUATION RESULTS ===\n')
eval_results=[]
for name,(preds,probs) in models_dict.items():
    eval_results.append({'Model':name,'Accuracy':accuracy_score(y_test,preds),
        'Precision':precision_score(y_test,preds),'Recall':recall_score(y_test,preds),
        'F1-Score':f1_score(y_test,preds),'AUC-ROC':roc_auc_score(y_test,probs)})
eval_df=pd.DataFrame(eval_results)
disp=eval_df.copy()
for c in ['Accuracy','Precision','Recall','F1-Score']: disp[c]=(disp[c]*100).round(2).astype(str)+'%'
disp['AUC-ROC']=disp['AUC-ROC'].round(4)
print(disp.to_string(index=False))
best_idx=eval_df['F1-Score'].idxmax()
print(f'\nBest (F1): {eval_df.loc[best_idx,"Model"]} - {eval_df.loc[best_idx,"F1-Score"]*100:.2f}%')

## Section 4: Confusion Matrices

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(16,10))
axes_flat=axes.flatten()
for i,(name,(preds,_)) in enumerate(models_dict.items()):
    cm=confusion_matrix(y_test,preds)
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=axes_flat[i],xticklabels=['Neg','Pos'],yticklabels=['Neg','Pos'])
    axes_flat[i].set_title(name);axes_flat[i].set_xlabel('Predicted');axes_flat[i].set_ylabel('Actual')
axes_flat[-1].axis('off')
plt.suptitle('Confusion Matrices',fontsize=14);plt.tight_layout()
plt.savefig('../models/confusion_matrices.png',dpi=150,bbox_inches='tight');plt.show()

## Section 5: ROC & Precision-Recall Curves

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
for name,(_,probs) in models_dict.items():
    fpr,tpr,_=roc_curve(y_test,probs);auc_val=roc_auc_score(y_test,probs)
    axes[0].plot(fpr,tpr,lw=2,label=f'{name} ({auc_val:.3f})')
axes[0].plot([0,1],[0,1],'k--');axes[0].set_title('ROC Curves');axes[0].set_xlabel('FPR');axes[0].set_ylabel('TPR');axes[0].legend(fontsize=8);axes[0].grid(alpha=0.3)
for name,(_,probs) in models_dict.items():
    p,r,_=precision_recall_curve(y_test,probs);axes[1].plot(r,p,lw=2,label=name)
axes[1].set_title('Precision-Recall Curves');axes[1].set_xlabel('Recall');axes[1].set_ylabel('Precision');axes[1].legend(fontsize=8);axes[1].grid(alpha=0.3)
plt.tight_layout();plt.savefig('../models/roc_pr_curves.png',dpi=150,bbox_inches='tight');plt.show()

## Section 6: Cross-Validation Stability

In [ ]:
print('=== 10-Fold Cross-Validation Stability ===\n')
cv10=StratifiedKFold(n_splits=10,shuffle=True,random_state=42)
cv_models={'LR':LogisticRegression(max_iter=1000,random_state=42,class_weight='balanced'),
    'RF':RandomForestClassifier(n_estimators=200,max_depth=20,random_state=42,class_weight='balanced',n_jobs=-1),
    'XGB':XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,scale_pos_weight=scale_weight,random_state=42,eval_metric='logloss',use_label_encoder=False),
    'GB':GradientBoostingClassifier(n_estimators=200,max_depth=5,random_state=42)}
cv_results={}
for name,model in cv_models.items():
    X_cv=X_combined_train if name=='LR' else X_train
    scores=cross_val_score(model,X_cv,y_train,cv=cv10,scoring='f1',n_jobs=-1)
    cv_results[name]=scores
    print(f'{name:5s}: F1={scores.mean()*100:.2f}% +/- {scores.std()*100:.2f}%')
fig,ax=plt.subplots(figsize=(8,5))
cv_df=pd.DataFrame(cv_results).melt(var_name='Model',value_name='F1')
sns.boxplot(data=cv_df,x='Model',y='F1',palette='Set2',ax=ax)
ax.set_title('CV F1 Distribution (10-Fold)');plt.tight_layout()
plt.savefig('../models/cv_stability.png',dpi=150,bbox_inches='tight');plt.show()

## Section 7: Statistical Significance

In [ ]:
from scipy.stats import ttest_rel
print('=== Paired t-test (CV Folds) ===\n')
names=list(cv_results.keys())
for i in range(len(names)):
    for j in range(i+1,len(names)):
        t,p=ttest_rel(cv_results[names[i]],cv_results[names[j]])
        sig='*' if p<0.05 else ''
        print(f'{names[i]} vs {names[j]}: t={t:.3f}, p={p:.4f} {sig}')
eval_df.to_csv('../models/evaluation_results.csv',index=False)
print('\nResults saved to ../models/evaluation_results.csv')